In [1]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

### Область работы 1 (библиотеки)

In [2]:
# Данный блок в области 1 выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
#
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
#
# pip install

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pandas.plotting as pd_plt
import matplotlib.colors as plt_colors
from IPython.display import display, display_html

from sklearn.linear_model import Ridge, SGDRegressor, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler, RobustScaler, QuantileTransformer, MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder, OneHotEncoder, TargetEncoder, KBinsDiscretizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, StratifiedShuffleSplit, StratifiedKFold
from sklearn.model_selection import validation_curve
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_transformer, TransformedTargetRegressor
from sklearn.base import BaseEstimator, TransformerMixin
import category_encoders as ce
from sklearn.datasets import make_regression
from sklearn.preprocessing import PolynomialFeatures


### Область работы 2 (выполнение лучшей модели)

In [4]:
# Данный блок(и) в области 2 выполняется преподавателем
#
# В области находится одна, единственная, итоговая модель машинного обучения с однозначными, 
# зафиксированными параметрами
#
# В данной области категорически запрещается искать, выбирать, улучшать, оптимизировать, 
# тюниговать и т.д. модель машинного обучения

In [4]:
# Путь к тренировочному набору
path_train = 'train.csv' # содержит только имя файла, без имен папок
# Путь к тестовому набору
path_test  = 'test.csv' # содержит только имя файла, без имен папок

In [5]:
df = pd.read_csv(path_train)
X_train = df.drop(['Weight'], axis=1)  
y_train = df['Weight']

X_test = pd.read_csv(path_test)

In [6]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BCS     164 non-null    float64
 1   Age     164 non-null    object 
 2   Sex     164 non-null    object 
 3   Length  164 non-null    int64  
 4   Girth   164 non-null    int64  
 5   Height  164 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 7.8+ KB


### Такой же Column Transformer, как и в tune_model (преобразовываем данные)

In [7]:
num = ['BCS', 'Length', 'Girth', 'Height']
cat = ['Age', 'Sex']

num_transformer = Pipeline(steps=[ 
    ('polynom', PolynomialFeatures(3, include_bias=False, interaction_only=True)),
    ('scaler', StandardScaler())])


cat_transformer = Pipeline(steps=[
    ('ce', OneHotEncoder(sparse_output=False)),
    ('scaler', StandardScaler())
])


 
CT = ColumnTransformer([
    ("num", num_transformer, num),
    ("cat", cat_transformer, cat)
],remainder="passthrough",verbose_feature_names_out= False).set_output(transform="pandas")

display(CT)

res_ct = CT.fit_transform(X_train)

res_ct

ColumnTransformer(remainder='passthrough',
                  transformers=[('num',
                                 Pipeline(steps=[('polynom',
                                                  PolynomialFeatures(degree=3,
                                                                     include_bias=False,
                                                                     interaction_only=True)),
                                                 ('scaler', StandardScaler())]),
                                 ['BCS', 'Length', 'Girth', 'Height']),
                                ('cat',
                                 Pipeline(steps=[('ce',
                                                  OneHotEncoder(sparse_output=False)),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Sex'])],
                  verbose_feature_names_out=False)

,BCS,Length,Girth,Height,BCS Length,BCS Girth,BCS Height,Length Girth,Length Height,Girth Height,...,Length Girth Height,Age_10-15,Age_15-20,Age_2-5,Age_5-10,Age_<2,Age_>20,Sex_female,Sex_gelding,Sex_stallion
0,0.22903,0.449042,-0.262243,-0.330818,0.345940,0.060085,0.104906,0.104898,0.167522,-0.340441,...,-0.035176,-0.909353,-0.247896,-0.462663,2.457038,-0.338186,-0.271010,-0.979164,-0.393720,1.287378
1,0.22903,-0.619102,0.902523,-1.004269,-0.106224,0.485828,-0.079888,0.039041,-0.865309,0.112552,...,-0.279466,-0.909353,-0.247896,-0.462663,2.457038,-0.338186,-0.271010,1.021279,-0.393720,-0.776773
2,0.22903,0.449042,0.902523,1.016084,0.345940,0.485828,0.474494,0.735853,0.739284,1.056928,...,0.888691,1.099683,-0.247896,-0.462663,-0.406994,-0.338186,-0.271010,-0.979164,2.539878,-0.776773
3,-0.93139,-0.485584,-1.427009,-1.228753,-0.916350,-1.193489,-1.127055,-1.046541,-0.860496,-1.466525,...,-1.181678,-0.909353,-0.247896,2.161398,-0.406994,-0.338186,-0.271010,1.021279,-0.393720,-0.776773
4,0.22903,-1.286693,-1.427009,-2.126688,-0.388826,-0.365657,-0.387879,-1.492670,-1.746055,-1.824825,...,-1.732027,-0.909353,-0.247896,-0.462663,-0.406994,2.956956,-0.271010,1.021279,-0.393720,-0.776773
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,0.22903,-0.218548,0.514268,0.567117,0.063338,0.343914,0.351298,0.100649,0.048164,0.580901,...,0.218463,-0.909353,-0.247896,-0.462663,-0.406994,-0.338186,3.689903,1.021279,-0.393720,-0.776773
376,0.22903,-3.690018,-3.109449,-3.473591,-1.406194,-0.980618,-0.757467,-3.457057,-3.732786,-3.316034,...,-3.399845,-0.909353,-0.247896,-0.462663,-0.406994,2.956956,-0.271010,-0.979164,-0.393720,1.287378
377,0.22903,1.116633,0.773105,-0.106334,0.628542,0.438523,0.166504,1.097714,0.748910,0.446112,...,0.835856,1.099683,-0.247896,-0.462663,-0.406994,-0.338186,-0.271010,1.021279,-0.393720,-0.776773
378,0.22903,0.716079,1.161360,0.342633,0.458981,0.580437,0.289700,1.053101,0.651691,0.917873,...,0.943458,1.099683,-0.247896,-0.462663,-0.406994,-0.338186,-0.271010,1.021279,-0.393720,-0.776773


In [6]:
# Блок обучения модели

### MSE в tune_model получался на тестовом наборе немного лучше в моделе Lasso, но я решила оставить Ridge, потому что он был стабильнее при повторных запусках (в решетке, где было несколько моделей видно, что std у 

In [8]:
model = Pipeline(steps=[('preproc', CT),
                            ('estimator', Ridge(max_iter=100000, alpha = 0.001))
                            ])
model.fit(X_train, y_train)

Pipeline(steps=[('preproc',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('polynom',
                                                                   PolynomialFeatures(degree=3,
                                                                                      include_bias=False,
                                                                                      interaction_only=True)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['BCS', 'Length', 'Girth',
                                                   'Height']),
                                                 ('cat',
                                                  Pipeline(steps=[('ce',
                                                                   OneHotEncoder(sparse_output=False)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Sex'])],
                                   verbose_feature_names_out=False)),
                ('estimator', Ridge(alpha=0.001, max_iter=100000))])

In [7]:
# Блок предсказания с использованием тестового набора

In [9]:
y_predict = model.predict(X_test)

In [10]:
# Вектора предсказанных значений  y_predict полученый на основане тестового набора
y_predict

array([146.42840142, 143.38592068, 161.01015142, 156.16464965,
       161.64144617, 122.38661205, 161.92962922, 153.67353451,
       141.2755097 , 175.07377367, 133.03353626, 153.59726075,
       200.8451133 , 135.00783107, 119.79328623, 170.50294535,
       156.82045311, 150.68914425, 144.43438219, 143.89646879,
       161.24995409, 157.94206811, 143.35963734, 157.69853265,
       173.65297214, 171.39674751, 155.45204023,  89.04955942,
       157.70005031, 160.51514033, 159.65258609, 154.89565103,
       196.37007597, 197.59420548, 126.88205965, 162.89178585,
       169.39746743, 179.78159138, 118.77272907, 175.63252346,
       161.25998429, 155.04648755, 171.5692307 , 146.14118289,
       162.01917368, 167.67199426, 139.12823426, 104.14705428,
       207.80195155, 153.31267779, 154.33147037, 150.83326629,
       181.92065696, 158.38385935, 179.24108707, 175.70834373,
       121.87589596, 163.75082749, 147.3041055 , 141.12176921,
       179.15388997,  96.12091912, 136.09395109, 167.09